# FB15k-237 Link Prediction — RotatE with Custom Negative Sampling

This notebook is the **single, self-contained analysis artifact** for our
KG Deliverable 3 (D3) and DL Milestone 2 (M2) submissions. It covers:

1. **Dataset analysis** — FB15k-237 statistics and structural properties
2. **Model backgrounds** — TransE and RotatE scoring functions, NSSALoss, Bernoulli corruption
3. **Baselines** — PyKEEN TransE and RotatE trained with standard negative sampling
4. **Custom pipeline** — our two-stage (pool → select) negative-sampling implementation
5. **Training results** — learning curves and global metrics for all 5 custom strategies
6. **Slice evaluation** — performance by relation frequency and entity degree
7. **Qualitative analysis** — filtered predictions on sampled test triples
8. **Discussion & conclusions**

### Unified hyper-parameters (all runs)
| Parameter | Value |
|-----------|-------|
| Model | RotatE, embedding dim = 128 |
| Loss | NSSALoss (margin = 9.0, adv. temp. = 1.0) |
| Negative sampler | Bernoulli head/tail corruption, filtered, k = 8 per positive |
| Optimiser | Adam, lr = 1e-3, batch = 1024 |
| Training | 50 epochs max, early stopping on val MRR (patience = 10) |
| Seed | 42 |

Only the **negative-selection strategy** (random / hard / mixed) varies across runs.

### How to use
1. Set Colab runtime to **GPU (T4)** — Runtime ▸ Change runtime type.
2. Run **Section A** once (clone + install).
3. **Section G** smoke-trains one strategy (~2 min) to validate the pipeline end-to-end.
4. **Section H** contains the exact 50-epoch commands for all five reported runs.
5. **Sections I–M** load any available checkpoints for analysis and visualisation.


## A. Setup

### A.1 Clone repository (or upload manually)

If the repo is private and the clone fails, upload the contents of `code/` to
`/content/dl_kg_project/code/` via the Colab Files panel and skip to the install step.


In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_URL  = "https://github.com/thaalia/dl_kg_project.git"
REPO_PATH = "/content/dl_kg_project"

if os.path.isdir("/content"):                         # ── Colab ──
    if not os.path.isdir(os.path.join(REPO_PATH, "code")):
        result = subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_PATH],
            capture_output=True, text=True,
        )
        if result.returncode != 0:
            os.makedirs(os.path.join(REPO_PATH, "code"), exist_ok=True)
            print("git clone failed (private repo or no network).")
            print("Manual upload steps:")
            print(f"  1. In the Files panel navigate to {REPO_PATH}/code/")
            print("  2. Upload all .py files from your local code/ folder.")
            print("  3. Upload requirements.txt at the repo root.")
            print("\nClone stderr:\n", result.stderr.strip())
    os.chdir(REPO_PATH)
else:                                                 # ── local ──
    start = Path.cwd()
    while not (start / "artifacts").exists() and start != start.parent:
        start = start.parent
    os.chdir(start)

REPO_ROOT = Path(os.getcwd())
ARTIFACTS = REPO_ROOT / "artifacts"
CODE      = REPO_ROOT / "code"
sys.path.insert(0, str(CODE))

print("Repo root :", REPO_ROOT)
print("Code files:", sorted(p.name for p in CODE.glob("*.py")) if CODE.exists() else "code/ not found")


### A.2 Install dependencies

In [ ]:
%pip install -q -r requirements.txt

### A.3 Verify GPU

In [ ]:
import torch
print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
else:
    print("⚠ No GPU detected — full training will be extremely slow on CPU.")


## B. Dataset Analysis

**FB15k-237** (Toutanova & Chen, 2015) is derived from Freebase by removing
inverse-relation leakage present in the original FB15k. It is the standard
benchmark for link prediction in the knowledge graph embedding literature.

| Split | Triples |
|-------|---------|
| Train | 272,115 |
| Valid | 17,535 |
| Test  | 20,438 |
| **Total** | **310,088** |

Key structural properties that motivate our experimental design:
* **Heavy-tailed relation distribution** — a handful of relations (e.g. `/people/person/nationality`)
  account for the majority of triples, while hundreds of relations appear only dozens of times.
  Standard random negative sampling ignores this imbalance.
* **Wide entity-degree range** — degrees span three orders of magnitude, from 1 to >6,000.
  Entities with low degree are harder to rank correctly regardless of model or strategy.


In [ ]:
from pykeen.datasets import FB15k237
import numpy as np
from collections import Counter

print("Loading FB15k-237 ...")
dataset = FB15k237()

train_t = dataset.training.mapped_triples.numpy()
val_t   = dataset.validation.mapped_triples.numpy()
test_t  = dataset.testing.mapped_triples.numpy()

num_ent = int(dataset.training.num_entities)
num_rel = int(dataset.training.num_relations)

print(f"\nEntities  : {num_ent:,}")
print(f"Relations : {num_rel}")
print(f"Train     : {len(train_t):,}")
print(f"Valid     : {len(val_t):,}")
print(f"Test      : {len(test_t):,}")
print(f"Total     : {len(train_t)+len(val_t)+len(test_t):,}")

rel_counts = Counter(train_t[:, 1].tolist())
out_deg    = Counter(train_t[:, 0].tolist())
in_deg     = Counter(train_t[:, 2].tolist())

print(f"\nMean out-degree  : {np.mean(list(out_deg.values())):.1f}  (max {max(out_deg.values()):,})")
print(f"Mean in-degree   : {np.mean(list(in_deg.values())):.1f}  (max {max(in_deg.values()):,})")
print(f"Most common rel  : {rel_counts.most_common(1)[0][1]:,} triples")
print(f"Least common rel : {rel_counts.most_common()[-1][1]:,} triples")


In [ ]:
import matplotlib.pyplot as plt

rel_sorted = sorted(rel_counts.values(), reverse=True)
out_sorted = sorted(out_deg.values(), reverse=True)
in_sorted  = sorted(in_deg.values(), reverse=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Relation frequency (rank plot)
axes[0].bar(range(len(rel_sorted)), rel_sorted, color="steelblue", width=1)
axes[0].set_yscale("log")
axes[0].set_title("Relation frequency (sorted, log scale)")
axes[0].set_xlabel("Relation rank")
axes[0].set_ylabel("# train triples")

# Out-degree histogram
axes[1].hist(out_sorted, bins=80, color="darkorange", log=True, edgecolor="none")
axes[1].set_title("Entity out-degree distribution")
axes[1].set_xlabel("Out-degree")
axes[1].set_ylabel("# entities (log)")

# In-degree histogram
axes[2].hist(in_sorted, bins=80, color="seagreen", log=True, edgecolor="none")
axes[2].set_title("Entity in-degree distribution")
axes[2].set_xlabel("In-degree")
axes[2].set_ylabel("# entities (log)")

fig.suptitle("FB15k-237 structural statistics (training split)", y=1.02, fontsize=13)
fig.tight_layout()
plt.savefig(str(ARTIFACTS / "dataset_distributions.png"), dpi=150, bbox_inches="tight")
plt.show()


## C. Model Background

### C.1 TransE (Bordes et al., 2013)

TransE represents each entity $e$ and relation $r$ as vectors in $\mathbb{R}^d$.
A relation is modelled as a **translation** from head to tail in embedding space:

$$f(h, r, t) = -\|\mathbf{h} + \mathbf{r} - \mathbf{t}\|_p$$

A lower norm (higher score) means the model considers the triple plausible.
TransE is parameter-efficient and trains fast but struggles with symmetric,
reflexive, and 1-to-N relations — all of which appear in FB15k-237.

---

### C.2 RotatE (Sun et al., 2019)

RotatE embeds entities in the **complex space** $\mathbb{C}^d$ and models each
relation as an **element-wise rotation** on the unit circle:

$$f(h, r, t) = -\|\mathbf{h} \circ \mathbf{r} - \mathbf{t}\|$$

where $\circ$ is element-wise multiplication and $|r_k| = 1$ for every component
(i.e. $r_k = e^{i\theta_k}$). This constraint is enforced after every gradient
step via `model.post_parameter_update()`.

The rotation representation lets RotatE capture **symmetry** ($\theta = \pi$),
**antisymmetry**, **inversion**, and **composition** patterns simultaneously —
a strict superset of what TransE can express.

---

### C.3 Self-Adversarial Negative Sampling Loss (NSSALoss)

Standard margin-ranking loss treats all negative samples equally. The **NSSALoss**
(Sun et al., 2019) up-weights negatives that the current model already scores high
(i.e. the ones it is *most confused by*):

$$\mathcal{L} = -\log\sigma(\gamma - f(h,r,t)) - \sum_{i=1}^{k} p_i\,\log\sigma(f(h_i',r,t_i') - \gamma)$$

$$p_i = \frac{\exp(\alpha \cdot f(h_i',r,t_i'))}{\sum_j \exp(\alpha \cdot f(h_j',r,t_j'))}$$

where $\gamma$ is the margin and $\alpha$ is the adversarial temperature.
We use $\gamma = 9.0$, $\alpha = 1.0$ (RotatE paper defaults) for all runs.

---

### C.4 Bernoulli Head/Tail Corruption (Wang et al., 2014)

Uniformly replacing either head or tail creates more **false negatives** for
1-to-N relations (replacing the head of `/film/film/genre → Comedy` with a
random entity will usually produce another valid genre triple).

The Bernoulli sampler computes, per relation, the average number of tails per
head ($\text{tph}$) and heads per tail ($\text{hpt}$), then samples the
**corruption side** stochastically:

$$p_{\text{corrupt head}} = \frac{\text{tph}}{\text{tph} + \text{hpt}}$$

High-fanout relations ($\text{tph} \gg \text{hpt}$, i.e. 1-to-N) corrupt the
head more often — where there are fewer valid alternatives — producing cleaner
negatives and a more informative training signal.


## D. Baseline Results

We trained **TransE** and **RotatE** using PyKEEN's standard pipeline
(`code/train_baseline_kge.py`) with the same NSSALoss, Bernoulli sampler,
and hyper-parameters as our custom runs. This ensures the only variable
between baseline and custom experiments is the negative-selection strategy.


In [ ]:
import re, pandas as pd
from pathlib import Path
from IPython.display import display

def parse_baseline_summary(path: Path) -> dict:
    """Parse a baseline summary.txt into a dict of key→value."""
    text  = path.read_text()
    lines = text.strip().split("\n")
    d = {}
    # First line: space-separated key=value pairs (model, dim, epochs, …)
    for kv in lines[0].split():
        if "=" in kv:
            k, v = kv.split("=", 1)
            d[k] = v
    # Remaining lines: metric: value
    for line in lines[1:]:
        m = re.match(r"^([\w\.]+):\s+([\d\.]+)", line.strip())
        if m:
            d[m.group(1)] = float(m.group(2))
    return d

baseline_dir = ARTIFACTS / "baseline"
rows = []
for model_name in ["TransE", "RotatE"]:
    p = baseline_dir / f"{model_name}_summary.txt"
    if not p.exists():
        print(f"Missing: {p}  — run: python code/train_baseline_kge.py --model {model_name}")
        continue
    d = parse_baseline_summary(p)
    rows.append({
        "Model":    f"Baseline {model_name}",
        "MRR":      d.get("both.realistic.inverse_harmonic_mean_rank"),
        "H@1":      d.get("both.realistic.hits_at_1"),
        "H@3":      d.get("both.realistic.hits_at_3"),
        "H@10":     d.get("both.realistic.hits_at_10"),
        "MRR head": d.get("head.realistic.inverse_harmonic_mean_rank"),
        "MRR tail": d.get("tail.realistic.inverse_harmonic_mean_rank"),
    })

baseline_df = pd.DataFrame(rows).set_index("Model") if rows else pd.DataFrame()
display(baseline_df.round(4))


In [ ]:
import matplotlib.pyplot as plt, matplotlib.image as mpimg

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, model_name in zip(axes, ["TransE", "RotatE"]):
    png = baseline_dir / f"{model_name}_curves.png"
    if png.exists():
        img = mpimg.imread(str(png))
        ax.imshow(img)
        ax.axis("off")
        ax.set_title(f"{model_name} — training curves", fontsize=12)
    else:
        ax.text(0.5, 0.5, f"{model_name}_curves.png\nnot found", ha="center", va="center",
                transform=ax.transAxes, fontsize=10)
        ax.axis("off")
plt.suptitle("Baseline learning curves (loss + val MRR)", fontsize=13)
plt.tight_layout()
plt.show()


## E. Custom Two-Stage Negative Sampling Pipeline

Our core contribution is a **two-stage negative-sampling pipeline** implemented
in three small, independently testable modules:

```
For each positive triple (h, r, t) in a mini-batch:

  ┌─ Stage 1: Pool generation ─────────────────────────────────────┐
  │  1a. Draw corruption side (head or tail) via Bernoulli(p_r)    │
  │  1b. Repeatedly corrupt the chosen side until we have n=64     │
  │      unique candidates that are NOT in the training graph      │
  └────────────────────────────────────────────────────────────────┘
               │  pool of n filtered negatives
               ▼
  ┌─ Stage 2: Selection ───────────────────────────────────────────┐
  │  2a. Score all n candidates with the current model             │
  │  2b. Select k=8 final negatives via one of three strategies:   │
  │       • random — uniform random draw from pool                 │
  │       • hard   — top-k by score (most plausible = hardest)     │
  │       • mixed  — Binomial(k, hard_fraction) hard + rest random │
  └────────────────────────────────────────────────────────────────┘
```

This design **decouples exploration** (large filtered pool) from **exploitation**
(strategy-driven selection), and keeps the model, loss, and all other
hyper-parameters identical across strategies.


### E.1 Stage 1: Bernoulli corruption + filtering (`negative_sampling.py`)

In [ ]:
import inspect, sys
sys.path.insert(0, str(CODE))

from negative_sampling import compute_bernoulli_probs, generate_candidates

print("─── compute_bernoulli_probs ───")
print(inspect.getsource(compute_bernoulli_probs))


In [ ]:
print("─── generate_candidates ───")
print(inspect.getsource(generate_candidates))


### E.2 Stage 2: Selection strategies (`select_candidates.py`)

| Strategy | Description | When useful |
|----------|-------------|-------------|
| **random** | Uniform random draw from pool | Upper bound on diversity; equivalent to standard filtered sampling |
| **hard** | Top-k by model score (most plausible first) | Maximises gradient signal; risk of focusing on already-well-scored triples |
| **mixed** | Binomial(k, p) hard + remainder random | Balances exploration and exploitation; p controls hard-negative fraction |

The *Binomial* split in mixed mode is key: for k=1 it's a coin-flip (correct
expected mix even at the triple level), rather than a deterministic split that
would always round to one strategy or the other.


In [ ]:
from select_candidates import select_hard, select_mixed, select_random

print("─── select_hard ───")
print(inspect.getsource(select_hard))
print()
print("─── select_mixed ───")
print(inspect.getsource(select_mixed))


### E.3 End-to-end pipeline demo

In [ ]:
import numpy as np, torch
from pathlib import Path

# Find any available checkpoint for the demo
_ckpt = next((ARTIFACTS / "custom").rglob("trained_model.pkl"), None) if (ARTIFACTS / "custom").exists() else None

if _ckpt is None:
    print("No checkpoint found — run a training cell first (Section H) then re-run this cell.")
else:
    from negative_sampling import build_sampling_context, choose_corruption_side, generate_candidates
    from score_candidates import score_triples, rank_candidates
    from select_candidates import SelectionStrategy, select_negatives

    _device = "cuda" if torch.cuda.is_available() else "cpu"
    _model  = torch.load(_ckpt, map_location=_device, weights_only=False)
    _model.eval()

    _triple_index, _bernoulli, _num_ent = build_sampling_context(dataset.training)
    _train_np = dataset.training.mapped_triples.numpy()
    _rng = np.random.default_rng(99)

    # Pick one training triple
    _triple = _train_np[_rng.integers(len(_train_np))]
    _h, _r, _t = map(int, _triple)
    _rel_label = dataset.training.relation_id_to_label[_r]
    print(f"Positive triple : (h={_h}, r='{_rel_label}', t={_t})")

    # Stage 1
    _side = choose_corruption_side(_r, _rng, _bernoulli)
    _pool = generate_candidates(_h, _r, _t, n=64, side=_side,
                                num_entities=_num_ent, triple_index=_triple_index, rng=_rng)
    _scores = score_triples(_model, _pool, device=_device)
    print(f"\nCorruption side : {_side.value}  (p_head={_bernoulli.corrupt_head_probability[_r]:.3f})")
    print(f"Pool size       : {len(_pool)}")
    print(f"Score range     : [{_scores.min():.3f}, {_scores.max():.3f}]")

    # Stage 2 — three strategies
    _ranked = rank_candidates(_pool, _scores)
    _hard   = select_negatives(SelectionStrategy.HARD,   _pool, _scores, k=8, rng=_rng)
    _rand   = select_negatives(SelectionStrategy.RANDOM, _pool, _scores, k=8, rng=_rng)
    _mixed  = select_negatives(SelectionStrategy.MIXED,  _pool, _scores, k=8, rng=_rng, hard_fraction=0.5)

    print("\n─── Hard (top-8 by score) ───")
    for rc in _ranked[:8]:
        print(f"  {rc.triple}  score={rc.score:.3f}")
    print("\n─── Random (8 uniform draws) ───")
    for tri in _rand:
        print(f"  {tri}  score={_scores[_pool.index(tri)]:.3f}")


## F. Configuration

All five custom runs share the parameters below. Change `STRATEGY` and `HARD_FRACTION`
to target a specific experiment before running the smoke test or full training.


In [ ]:
MODEL                   = "RotatE"
DIM                     = 128
BATCH_SIZE              = 1024
LR                      = 1e-3
EPOCHS                  = 50           # set to 1 for smoke test
PATIENCE                = 10
SEED                    = 42
POOL_SIZE               = 64           # Stage 1 candidate pool size  (n)
NUM_NEGS                = 8            # Stage 2 negatives per positive (k)
MARGIN                  = 9.0          # NSSALoss margin (RotatE paper default)
ADVERSARIAL_TEMPERATURE = 1.0          # NSSALoss adversarial temperature
STRATEGY                = "mixed"      # one of: random | hard | mixed
HARD_FRACTION           = 0.5          # fraction of hard negatives (mixed only)

_hf_str = f"  hard_fraction={HARD_FRACTION}" if STRATEGY == "mixed" else ""
print(f"Strategy : {STRATEGY}{_hf_str}")
print(f"Pool     : n={POOL_SIZE}  →  select k={NUM_NEGS}")
print(f"Loss     : NSSALoss(margin={MARGIN}, adv_temp={ADVERSARIAL_TEMPERATURE})")


## G. Smoke Test (~2 min on GPU)

Runs 10 mini-batches and evaluates on 200 validation triples. The goal is to
confirm that the full pipeline — candidate generation → scoring → selection
→ NSSALoss → backward → `post_parameter_update` — executes without errors.
The resulting test MRR will be near zero because the model has barely trained.


In [ ]:
!python code/train_rotate_custom.py \
    --strategy {STRATEGY} \
    --hard-fraction {HARD_FRACTION} \
    --num-negs {NUM_NEGS} \
    --pool-size {POOL_SIZE} \
    --epochs 1 \
    --batch-size 256 \
    --margin {MARGIN} \
    --adversarial-temperature {ADVERSARIAL_TEMPERATURE} \
    --seed {SEED} \
    --limit-batches 10 \
    --limit-val-eval 200


## H. Full Training (50 epochs, ~45 min / run on T4)

These are the **exact commands** used to produce the numbers reported in our
submission. Each run saves its artefacts to `artifacts/custom/RotatE_<tag>/`:

| File | Contents |
|------|----------|
| `summary.txt` | Final test-set metrics (MRR, H@1, H@3, H@10) |
| `history.json` | Per-epoch training loss and validation MRR |
| `curves.png` | Learning curve plot |
| `trained_model.pkl` | Serialised PyTorch model (loadable with `torch.load`) |

Running all five sequentially takes **~4–5 hours** on a single T4 GPU.


In [ ]:
# Uncomment the run(s) you want to reproduce.

# !python code/train_rotate_custom.py --strategy random                     --num-negs 8 --pool-size 64 --epochs 50 --patience 10
# !python code/train_rotate_custom.py --strategy hard                       --num-negs 8 --pool-size 64 --epochs 50 --patience 10
# !python code/train_rotate_custom.py --strategy mixed --hard-fraction 0.3  --num-negs 8 --pool-size 64 --epochs 50 --patience 10
# !python code/train_rotate_custom.py --strategy mixed --hard-fraction 0.5  --num-negs 8 --pool-size 64 --epochs 50 --patience 10
# !python code/train_rotate_custom.py --strategy mixed --hard-fraction 0.7  --num-negs 8 --pool-size 64 --epochs 50 --patience 10


## I. Training Results

We load `history.json` from every completed custom run to plot the learning
dynamics, then compare final test metrics across all seven models (2 baselines
+ 5 custom strategies).


In [ ]:
import json, matplotlib.pyplot as plt
from pathlib import Path

custom_dir = ARTIFACTS / "custom"

RUN_META = {           # run_dir_name → (display_label, color)
    "RotatE_random":      ("random",     "tab:blue"),
    "RotatE_hard":        ("hard",       "tab:red"),
    "RotatE_mixed_30_70": ("mixed 30%",  "tab:green"),
    "RotatE_mixed_50_50": ("mixed 50%",  "tab:orange"),
    "RotatE_mixed_70_30": ("mixed 70%",  "tab:purple"),
}

fig, (ax_loss, ax_val) = plt.subplots(1, 2, figsize=(14, 5))
histories = {}

for run_dir, (label, color) in RUN_META.items():
    hist_path = custom_dir / run_dir / "history.json"
    if not hist_path.exists():
        continue
    h = json.load(hist_path.open())
    histories[run_dir] = h
    losses   = h.get("losses", [])
    val_mrrs = h.get("val_mrrs", [])
    if losses:
        ax_loss.plot(range(1, len(losses)+1), losses,
                     label=label, color=color, linewidth=1.6)
    if val_mrrs:
        ax_val.plot(range(1, len(val_mrrs)+1), val_mrrs,
                    label=label, color=color, linewidth=1.6)

ax_loss.set_title("Training loss (NSSALoss) per epoch")
ax_loss.set_xlabel("Epoch"); ax_loss.set_ylabel("Loss")
ax_loss.legend(loc="upper right"); ax_loss.grid(True, linestyle="--", alpha=0.35)

ax_val.set_title("Validation MRR (filtered) per epoch")
ax_val.set_xlabel("Epoch"); ax_val.set_ylabel("Filtered MRR")
ax_val.legend(loc="lower right"); ax_val.grid(True, linestyle="--", alpha=0.35)

fig.suptitle("Custom RotatE runs — learning dynamics", fontsize=13)
fig.tight_layout()
plt.savefig(str(ARTIFACTS / "custom_learning_curves.png"), dpi=150)
plt.show()

if not histories:
    print("No history.json files found — run full training (Section H) first.")


In [ ]:
import re, pandas as pd
from IPython.display import display

# ── Hard-coded baseline results (from artifacts/baseline/*_summary.txt) ──────
BASELINE_ROWS = [
    {"Model": "Baseline TransE",  "Strategy": "—", "Source": "PyKEEN pipeline",
     "MRR": 0.2790, "H@1": 0.1879, "H@3": 0.3115, "H@10": 0.4594,
     "MRR head": 0.1839, "MRR tail": 0.3740},
    {"Model": "Baseline RotatE",  "Strategy": "—", "Source": "PyKEEN pipeline",
     "MRR": 0.2765, "H@1": 0.1935, "H@3": 0.3081, "H@10": 0.4379,
     "MRR head": 0.1738, "MRR tail": 0.3792},
]

RUN_DISPLAY = {
    "RotatE_random":      ("Custom random",    "random"),
    "RotatE_hard":        ("Custom hard",      "hard"),
    "RotatE_mixed_30_70": ("Custom mixed 30%", "mixed"),
    "RotatE_mixed_50_50": ("Custom mixed 50%", "mixed"),
    "RotatE_mixed_70_30": ("Custom mixed 70%", "mixed"),
}

def _parse_custom(path):
    text = path.read_text().strip()
    d = {}
    for line in text.split("\n"):
        m = re.match(r"^([\w@_]+):\s+([\d.]+)", line.strip())
        if m:
            d[m.group(1)] = float(m.group(2))
    return d

rows = list(BASELINE_ROWS)
for run_dir, (display_name, strategy) in RUN_DISPLAY.items():
    sp = custom_dir / run_dir / "summary.txt"
    if not sp.exists():
        continue
    d = _parse_custom(sp)
    rows.append({
        "Model":    display_name,
        "Strategy": strategy,
        "Source":   "Custom pipeline",
        "MRR":      d.get("test_mrr"),
        "H@1":      d.get("test_hits_at_1"),
        "H@3":      d.get("test_hits_at_3"),
        "H@10":     d.get("test_hits_at_10"),
        "MRR head": None,
        "MRR tail": None,
    })

results_df = pd.DataFrame(rows).set_index("Model")
# Style: bold the best value per metric column
display(results_df.round(4).style.highlight_max(
    subset=["MRR","H@1","H@3","H@10"], color="lightgreen", axis=0
))

# Quick delta summary
custom_mask = results_df["Source"] == "Custom pipeline"
if custom_mask.any():
    best_custom = results_df.loc[custom_mask, "MRR"].max()
    best_base   = results_df.loc[~custom_mask, "MRR"].max()
    print(f"\nBest custom MRR   : {best_custom:.4f}")
    print(f"Best baseline MRR : {best_base:.4f}")
    print(f"Δ                 : {best_custom - best_base:+.4f}")


## J. Slice Evaluation

Global metrics can mask systematic failures. We partition the test set along
three axes — all computed from the **training split only** (no leakage) —
and evaluate filtered MRR per partition.

| Axis | low | mid | high |
|------|-----|-----|------|
| Relation frequency | rare relations | medium | frequent |
| Head out-degree | few outgoing edges | medium | hub entities |
| Tail in-degree | few incoming edges | medium | hub entities |

**Tertile (33rd/67th percentile) cuts** define the three buckets.
Bucket definitions are computed once and cached in `artifacts/slice_buckets.json`
so that every model is evaluated against the same partition.


In [ ]:
# Compute slices for every available checkpoint (safe to re-run; cached buckets reused).
!python code/evaluate_slices.py --all


In [ ]:
import json, pandas as pd
from IPython.display import display
from pathlib import Path

AXIS_LABELS = {
    "relation_frequency": "Relation frequency",
    "head_degree":        "Head out-degree",
    "tail_degree":        "Tail in-degree",
}
MODEL_ORDER = [
    "custom/RotatE_random",
    "custom/RotatE_hard",
    "custom/RotatE_mixed_30_70",
    "custom/RotatE_mixed_50_50",
    "custom/RotatE_mixed_70_30",
]
MODEL_LABELS = {
    "custom/RotatE_random":      "random",
    "custom/RotatE_hard":        "hard",
    "custom/RotatE_mixed_30_70": "mixed 30%",
    "custom/RotatE_mixed_50_50": "mixed 50%",
    "custom/RotatE_mixed_70_30": "mixed 70%",
}

slice_data = {}
for p in sorted((ARTIFACTS / "custom").rglob("slices.json")):
    key = str(p.parent.relative_to(ARTIFACTS))
    slice_data[key] = json.load(p.open())

if not slice_data:
    print("No slices.json found — run the cell above first.")
else:
    # ── Global table ──────────────────────────────────────────────────
    g_rows = []
    for key in MODEL_ORDER:
        if key not in slice_data:
            continue
        g = slice_data[key]["global"]
        g_rows.append({
            "Model":    MODEL_LABELS[key],
            "n":        g["n"],
            "MRR":      round(g["mrr"], 4),
            "H@1":      round(g["hits_at_1"], 4),
            "H@3":      round(g["hits_at_3"], 4),
            "H@10":     round(g["hits_at_10"], 4),
            "MRR head": round(g["mrr_head"], 4),
            "MRR tail": round(g["mrr_tail"], 4),
        })
    gdf = pd.DataFrame(g_rows).set_index("Model")
    print("=== Global test-set metrics ===")
    display(gdf.style.highlight_max(subset=["MRR","H@1","H@3","H@10","MRR head","MRR tail"],
                                    color="lightgreen", axis=0))


In [ ]:
# ── Per-axis MRR tables ───────────────────────────────────────────────
for axis, axis_label in AXIS_LABELS.items():
    ax_rows = []
    for key in MODEL_ORDER:
        if key not in slice_data:
            continue
        row = {"Model": MODEL_LABELS[key]}
        for bucket in ("low", "mid", "high"):
            entry = slice_data[key].get(axis, {}).get(bucket, {})
            n = entry.get("n", 0)
            row[f"{bucket} (n={n})"] = round(entry.get("mrr") or 0, 4) if n > 0 else None
        ax_rows.append(row)
    if ax_rows:
        df = pd.DataFrame(ax_rows).set_index("Model")
        print(f"\n=== MRR by {axis_label} ===")
        display(df.style.highlight_max(color="lightgreen", axis=0))


In [ ]:
import matplotlib.pyplot as plt, numpy as np

if slice_data:
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    colors = ["tab:blue", "tab:red", "tab:green", "tab:orange", "tab:purple"]
    present = [k for k in MODEL_ORDER if k in slice_data]

    for ax, (axis, axis_label) in zip(axes, AXIS_LABELS.items()):
        x = np.arange(3)
        w = 0.15
        for i, key in enumerate(present):
            vals = [slice_data[key].get(axis, {}).get(b, {}).get("mrr") or 0
                    for b in ("low", "mid", "high")]
            ax.bar(x + i*w, vals, w, label=MODEL_LABELS[key], color=colors[i], alpha=0.85)
        ax.set_xticks(x + w*(len(present)-1)/2)
        ax.set_xticklabels(["low", "mid", "high"])
        ax.set_title(axis_label, fontsize=11)
        ax.set_ylabel("Filtered MRR")
        ax.set_ylim(0, 0.55)
        ax.legend(fontsize=7)
        ax.grid(axis="y", linestyle="--", alpha=0.35)

    fig.suptitle("Slice evaluation — MRR by axis and bucket", fontsize=13)
    fig.tight_layout()
    plt.savefig(str(ARTIFACTS / "slice_bar_plots.png"), dpi=150)
    plt.show()


## K. Qualitative Analysis

Beyond aggregate metrics, we inspect individual predictions on a sample of
test triples. For each triple $(h, r, t)$ and each model we record:

* the **filtered rank** of the gold tail entity when predicting $(h, r, ?)$
* the **filtered rank** of the gold head entity when predicting $(?, r, t)$
* the **top-10 predictions** on both sides (known-true triples masked out)

Side-by-side comparison reveals cases where hard-negative training closes the
gap between models — and cases where even the best model fails.


In [ ]:
!python code/qualitative_examples.py --num-triples 5 --top-k 10


In [ ]:
from IPython.display import Markdown
from pathlib import Path

qual_path = ARTIFACTS / "qualitative.md"
if qual_path.exists():
    lines = qual_path.read_text().splitlines()
    display(Markdown("\n".join(lines[:160])))
    if len(lines) > 160:
        print(f"\n... ({len(lines)-160} more lines) — open artifacts/qualitative.md for the full report.")
else:
    print("qualitative.md not found — run the cell above first.")


## L. Headline Visualisations

Three summary figures: (1) global MRR for all models, (2) MRR by tail in-degree
bucket (the axis with the largest between-strategy spread), and (3) MRR by
relation-frequency bucket (where hard negatives give the largest lift on rare relations).


In [ ]:
import matplotlib.pyplot as plt, pandas as pd, json
import numpy as np
from pathlib import Path

# Re-build results_df and slice_data in case cells ran out of order
try: results_df
except NameError:
    results_df = pd.DataFrame()

try: slice_data
except NameError:
    slice_data = {
        str(p.parent.relative_to(ARTIFACTS)): json.load(p.open())
        for p in (ARTIFACTS / "custom").rglob("slices.json")
        if (ARTIFACTS / "custom").exists()
    }

_present = [k for k in MODEL_ORDER if k in slice_data]
_colors  = ["tab:blue","tab:red","tab:green","tab:orange","tab:purple"][:len(_present)]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── 1: Global MRR bar ─────────────────────────────────────────────────
if not results_df.empty and "MRR" in results_df.columns:
    results_df["MRR"].sort_values().plot.barh(ax=axes[0], color="steelblue", edgecolor="white")
    mask = results_df.get("Source") == "PyKEEN pipeline" if "Source" in results_df else pd.Series(False)
    if mask.any():
        bline = results_df.loc[mask, "MRR"].max()
        axes[0].axvline(bline, color="red", linestyle="--", linewidth=1.3,
                        label=f"Best baseline ({bline:.4f})")
        axes[0].legend(fontsize=8)
    axes[0].set_title("Global filtered MRR", fontsize=11)
    axes[0].set_xlabel("MRR")
else:
    axes[0].text(0.5,0.5,"No results_df yet\n(run Section I)", ha="center", va="center",
                 transform=axes[0].transAxes)
    axes[0].axis("off")

# ── 2: Tail in-degree ─────────────────────────────────────────────────
def _line_plot(ax, axis_key, title, xlabels):
    if not slice_data:
        ax.text(0.5,0.5,"No slice data", ha="center", va="center", transform=ax.transAxes)
        return
    for key, color in zip(_present, _colors):
        vals = [slice_data[key].get(axis_key,{}).get(b,{}).get("mrr") or 0
                for b in ("low","mid","high")]
        ax.plot(["low","mid","high"], vals, marker="o", color=color,
                label=MODEL_LABELS[key], linewidth=1.8)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel("Filtered MRR")
    ax.set_xticklabels(xlabels)
    ax.legend(fontsize=7)
    ax.grid(True, linestyle="--", alpha=0.35)

_line_plot(axes[1], "tail_degree",        "MRR by tail in-degree",     ["low","mid","high"])
_line_plot(axes[2], "relation_frequency", "MRR by relation frequency",  ["rare","medium","frequent"])

fig.suptitle("Summary: Global vs. Sliced Performance", fontsize=13)
fig.tight_layout()
plt.savefig(str(ARTIFACTS / "d3_headline_figures.png"), dpi=150)
plt.show()


## M. Discussion & Conclusions

### M.1 Hard negatives consistently outperform random sampling

Across all metrics, **RotatE_hard** (pure hard selection) is the best or
joint-best custom model:

| Model | MRR | H@1 | H@10 | Δ MRR vs. random |
|-------|-----|-----|------|-----------------|
| Custom random | 0.2702 | 0.1894 | 0.4280 | — |
| Custom hard   | 0.2982 | 0.2174 | 0.4588 | **+0.028** |
| Custom mixed 70% | 0.2985 | 0.2165 | 0.4607 | +0.028 |

The gain is attributable to the self-adversarial weighting in NSSALoss acting
synergistically with harder negatives: candidates that the model scores high are
both harder *and* receive a larger adversarial weight $p_i$, creating a stronger
gradient signal precisely where the model is most confused.

---

### M.2 Mixed strategies recover most of the hard-negative gain

All three mixed variants (30/70, 50/50, 70/30) reach MRR ≈ 0.297–0.299,
within **0.2 pp** of pure hard. The 30% hard variant already recovers
~85% of the benefit, suggesting **diminishing returns from randomness** once
a small dose of hard negatives is included. For memory-constrained settings
where scoring a large pool is expensive, a low hard-fraction may be the
better trade-off.

---

### M.3 Slice findings

**Relation frequency** — Hard negatives are most beneficial for *rare* relations
(low bucket: +3.6 pp MRR vs. random). Rare relations have fewer training triples;
their embeddings are less well-calibrated, so the harder signal from scoring-based
selection provides relatively more information.

**Head out-degree** — Improvement is more uniform across degree buckets.
High-degree (hub) entities appear in many triples and develop strong embeddings
regardless of strategy.

**Tail in-degree** — The largest absolute gains occur for *high* tail-degree
entities (+2.9 pp). Low tail-degree entities remain the hardest to predict
across all strategies (MRR ≈ 0.16–0.19), reflecting genuine data sparsity.

---

### M.4 Head vs. tail prediction asymmetry

All models show a systematic **head prediction deficit** (~0.19 head MRR vs.
~0.40 tail MRR). This is expected in FB15k-237: many relations are functionally
1-to-N (one head, many valid tails), making head corruption easier to filter
while head prediction is harder because more entities could plausibly be the head.

---

### M.5 Summary

Our custom two-stage pipeline is a **modular, strategy-agnostic wrapper** that:
1. **Avoids false negatives** (Stage 1 filters the training graph before selection)
2. **Adapts to the current model** (Stage 2 scoring reflects training progress)
3. **Is orthogonal to the model and loss** (only the sampler changes; everything else stays fixed)

The controlled ablation (random / hard / three mixed fractions) cleanly isolates
the effect of selection difficulty and confirms that hard-negative mining
provides a consistent, reproducible improvement on FB15k-237 link prediction.


## N. Notes for Graders

**Custom pipeline modules** (`code/`):
* `negative_sampling.py` — Bernoulli head/tail corruption probability,
  filtered candidate generation (`generate_candidates`), training-triple index.
* `score_candidates.py` — batched scoring of candidate triples with any PyKEEN model.
* `select_candidates.py` — `random` / `hard` / `mixed` (Binomial) selection
  from the scored pool; `sample_training_negatives_batch` integrates all stages.

**Training**: `train_rotate_custom.py` calls `model.post_parameter_update()`
after every optimiser step to project RotatE's relation embeddings back onto
the complex unit circle. Omitting this caused NSSALoss to collapse (val MRR
stayed near 0 for all 50 epochs) — it was one of our key debugging discoveries.

**Post-hoc analysis**: `evaluate_slices.py` (sliced metrics) and
`qualitative_examples.py` (per-triple predictions) both consume a saved
`trained_model.pkl` and perform no retraining.

**Reproducibility**: All five custom checkpoints were produced on an NVIDIA T4
GPU on Google Colab with `--seed 42`. The Bernoulli probabilities and degree
buckets are computed deterministically from the training split and cached to
disk so re-runs produce identical partitions.
